# QICK Pre-Flight Checklist
### RFSoC 4x2 — High-Resolution RFI Monitor
**Run every cell top to bottom before starting any observation.**  
Every cell prints either ✅ PASS or ❌ FAIL with a clear explanation.  
Do not proceed to the main survey script unless all cells show PASS.

---

## Cell 1 — Python environment and imports

In [ ]:
import sys, os, time, datetime

REQUIRED = ['numpy', 'matplotlib', 'qick']
missing  = []

for pkg in REQUIRED:
    try:
        __import__(pkg)
        print(f'  ✅ {pkg} importable')
    except ImportError:
        print(f'  ❌ {pkg} NOT FOUND')
        missing.append(pkg)

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

print(f'\nPython  : {sys.version.split()[0]}')
print(f'NumPy   : {np.__version__}')
print(f'Matplotlib: {matplotlib.__version__}')

if missing:
    print(f'\n❌ FAIL — missing packages: {missing}')
    print('   Run: pip install', ' '.join(missing))
else:
    print('\n✅ PASS — all required packages available')

## Cell 2 — PYNQ version check

In [ ]:
import pkg_resources

try:
    pynq_ver = pkg_resources.get_distribution('pynq').version
    print(f'  PYNQ version: {pynq_ver}')

    major, minor = int(pynq_ver.split('.')[0]), int(pynq_ver.split('.')[1])

    if major == 3 and minor == 0:
        print('  ✅ PASS — PYNQ 3.0.x confirmed (QICK compatible)')
    elif major >= 3 and minor >= 1:
        print('  ❌ FAIL — PYNQ 3.1+ detected')
        print('   QICK 0.2.388 requires PYNQ < 3.1')
        print('   Reflash SD card with PYNQ 3.0.1 image before proceeding')
    else:
        print(f'  ⚠️  WARNING — unexpected PYNQ version {pynq_ver}')
        print('   Tested with PYNQ 3.0.1 — proceed with caution')

except Exception as e:
    print(f'  ❌ FAIL — could not read PYNQ version: {e}')

## Cell 3 — QICK version check

In [ ]:
try:
    import qick
    qick_ver = pkg_resources.get_distribution('qick').version
    print(f'  QICK version: {qick_ver}')

    major = int(qick_ver.split('.')[0])
    minor = int(qick_ver.split('.')[1])

    if major == 0 and minor == 2:
        print('  ✅ PASS — QICK 0.2.x confirmed')
    else:
        print(f'  ⚠️  WARNING — untested QICK version {qick_ver}')
        print('   This notebook was validated on QICK 0.2.388')

except Exception as e:
    print(f'  ❌ FAIL — could not import QICK: {e}')
    print('   Run: pip install --no-index --find-links=/home/xilinx/qick_packages qick')

## Cell 4 — Bitstream files present on board

In [ ]:
BIT_FILE = '/home/xilinx/qick_repo/qick_lib/qick/qick_4x2.bit'
HWH_FILE = '/home/xilinx/qick_repo/qick_lib/qick/qick_4x2.hwh'

passed = True

for fpath in [BIT_FILE, HWH_FILE]:
    if os.path.exists(fpath):
        size_mb = os.path.getsize(fpath) / 1e6
        print(f'  ✅ Found: {os.path.basename(fpath)} ({size_mb:.1f} MB)')
    else:
        print(f'  ❌ MISSING: {fpath}')
        passed = False

if passed:
    print('\n✅ PASS — bitstream files confirmed')
else:
    print('\n❌ FAIL — copy qick_repo to the board first:')
    print('   scp -r ~/Downloads/qick_repo/* xilinx@192.168.3.1:/home/xilinx/qick_repo/')

## Cell 5 — Load QICK overlay onto FPGA
**This takes about 60 seconds. The FPGA is being programmed.**

In [ ]:
from qick import QickSoc

print('[LOAD] Programming FPGA with QICK bitstream...')
print('[LOAD] This takes ~60 seconds — please wait\n')

t0 = time.time()
try:
    soc = QickSoc(bitfile=BIT_FILE)
    load_time = time.time() - t0
    print(f'\n[LOAD] Completed in {load_time:.1f}s')
    print('\n✅ PASS — QICK overlay loaded successfully')
    print('\n--- QICK hardware configuration ---')
    print(soc)
except Exception as e:
    print(f'\n❌ FAIL — overlay load error: {e}')
    print('   Check bitstream file path and PYNQ version')
    soc = None

## Cell 6 — Hardware configuration validation

In [ ]:
if soc is None:
    print('❌ SKIP — soc not loaded, fix Cell 5 first')
else:
    passed = True

    # Expected values confirmed from previous validated run
    EXPECTED_ADC_FS   = 4423.680   # MHz
    EXPECTED_DAC_FS   = 9830.400   # MHz
    EXPECTED_N_RO     = 2          # readout channels
    EXPECTED_N_GEN    = 2          # signal generator channels
    FS_TOLERANCE_MHZ  = 1.0        # allow 1 MHz tolerance

    # Check readout channels
    n_ro = len(soc.config['readouts'])
    if n_ro == EXPECTED_N_RO:
        print(f'  ✅ Readout channels: {n_ro} (expected {EXPECTED_N_RO})')
    else:
        print(f'  ❌ Readout channels: {n_ro} (expected {EXPECTED_N_RO})')
        passed = False

    # Check signal generators
    n_gen = len(soc.config['gens'])
    if n_gen == EXPECTED_N_GEN:
        print(f'  ✅ Signal generators: {n_gen} (expected {EXPECTED_N_GEN})')
    else:
        print(f'  ❌ Signal generators: {n_gen} (expected {EXPECTED_N_GEN})')
        passed = False

    # Check ADC sample rate
    adc_fs = soc.config['readouts'][0]['fs']
    if abs(adc_fs - EXPECTED_ADC_FS) < FS_TOLERANCE_MHZ:
        print(f'  ✅ ADC sample rate: {adc_fs:.3f} MHz (expected {EXPECTED_ADC_FS} MHz)')
    else:
        print(f'  ❌ ADC sample rate: {adc_fs:.3f} MHz (expected {EXPECTED_ADC_FS} MHz)')
        passed = False

    # Check DAC sample rate
    dac_fs = soc.config['gens'][0]['fs']
    if abs(dac_fs - EXPECTED_DAC_FS) < FS_TOLERANCE_MHZ:
        print(f'  ✅ DAC sample rate: {dac_fs:.3f} MHz (expected {EXPECTED_DAC_FS} MHz)')
    else:
        print(f'  ❌ DAC sample rate: {dac_fs:.3f} MHz (expected {EXPECTED_DAC_FS} MHz)')
        passed = False

    # Check DDR4 buffer exists
    has_ddr4 = hasattr(soc, 'arm_ddr4') and hasattr(soc, 'get_ddr4')
    if has_ddr4:
        print(f'  ✅ DDR4 API available (arm_ddr4, get_ddr4)')
    else:
        print(f'  ❌ DDR4 API NOT available — arm_ddr4 or get_ddr4 missing')
        print('     This QICK version may not support DDR4 capture')
        passed = False

    # Report computed parameters
    FS_MHZ   = adc_fs
    NYQUIST  = FS_MHZ / 2
    DF_1M    = FS_MHZ * 1e3 / 1_048_576
    DF_512K  = FS_MHZ * 1e3 / 524_288

    print(f'\n--- Computed spectral parameters ---')
    print(f'  ADC fs           : {FS_MHZ:.3f} MHz')
    print(f'  Nyquist          : {NYQUIST:.3f} MHz')
    print(f'  Resolution @1M   : {DF_1M:.2f} kHz/bin  (Phil target: <10 kHz) ✅')
    print(f'  Resolution @512k : {DF_512K:.2f} kHz/bin (faster, borderline)  ⚠️')

    print(f'\n{"✅ PASS" if passed else "❌ FAIL"} — hardware configuration check')

## Cell 7 — DDR4 API smoke test
Confirms `arm_ddr4` and `get_ddr4` work and return data of the expected shape.

In [ ]:
from qick.averager_program import AveragerProgram

if soc is None or not has_ddr4:
    print('❌ SKIP — soc not loaded or DDR4 API unavailable')
else:
    # Small test capture — 256 transfers = 65,536 samples
    TEST_NT = 256

    class MinimalTrigger(AveragerProgram):
        def initialize(self):
            self.declare_readout(ch=0, length=1000, freq=0, gen_ch=None)
            self.synci(200)
        def body(self):
            self.trigger(adcs=[0], pins=[0], adc_trig_offset=100)
            self.wait_all()
            self.sync_all(self.us2cycles(1.0))

    prog = MinimalTrigger(soc, {'ro_ch':0,'readout_length':1000,
                                 'adc_trig_offset':100,'soft_avgs':1,
                                 'reps':1,'relax_delay':1.0})
    try:
        print(f'  [DDR4] Arming buffer with nt={TEST_NT}...')
        soc.arm_ddr4(ch=0, nt=TEST_NT)

        print(f'  [DDR4] Running trigger program...')
        soc.run_rounds(prog, rounds=1)

        print(f'  [DDR4] Retrieving data...')
        raw = soc.get_ddr4(ch=0, nt=TEST_NT, start=None)

        print(f'  [DDR4] Raw data shape : {raw.shape}')
        print(f'  [DDR4] Data type      : {raw.dtype}')
        print(f'  [DDR4] I range        : {raw[:,0].min():.1f} to {raw[:,0].max():.1f}')
        print(f'  [DDR4] Q range        : {raw[:,1].min():.1f} to {raw[:,1].max():.1f}')

        # Validate
        if raw.shape[0] > 0 and raw.shape[1] == 2:
            print('\n  ✅ PASS — DDR4 capture returns valid IQ data')
            DDR4_SAMPLES_PER_TRANSFER = raw.shape[0] // TEST_NT
            print(f'  [INFO] Samples per transfer: {DDR4_SAMPLES_PER_TRANSFER}')
            print(f'  [INFO] Use this to compute NT for your FFT size')
        else:
            print('\n  ❌ FAIL — unexpected DDR4 data shape')

    except Exception as e:
        print(f'\n  ❌ FAIL — DDR4 capture error: {e}')
        print('   This may mean DDR4 is not wired to readout ch 0 in this firmware')
        DDR4_SAMPLES_PER_TRANSFER = None

## Cell 8 — Decimated buffer smoke test
Confirms the standard `acquire_decimated` path still works as a fallback.

In [ ]:
if soc is None:
    print('❌ SKIP — soc not loaded')
else:
    class QuickCapture(AveragerProgram):
        def initialize(self):
            self.declare_readout(ch=0, length=self.cfg['readout_length'],
                                  freq=0, gen_ch=None)
            self.synci(200)
        def body(self):
            self.trigger(adcs=[0], pins=[0],
                          adc_trig_offset=self.cfg['adc_trig_offset'])
            self.wait_all()
            self.sync_all(self.us2cycles(self.cfg['relax_delay']))

    cfg = {'ro_ch':0,'readout_length':1000,'adc_trig_offset':100,
           'soft_avgs':1,'reps':1,'relax_delay':1.0}
    try:
        p   = QuickCapture(soc, cfg)
        iq  = p.acquire_decimated(soc, load_pulses=False, progress=False)
        i_d = iq[0][0]
        q_d = iq[0][1]

        print(f'  Decimated I samples : {len(i_d)}')
        print(f'  Decimated I range   : {i_d.min():.1f} to {i_d.max():.1f}')

        if len(i_d) > 0:
            print('\n  ✅ PASS — decimated buffer working')
            print('  [INFO] Decimated fs =',
                  soc.config['readouts'][0]['fs_decimated'], 'MHz')
        else:
            print('\n  ❌ FAIL — empty decimated buffer')
    except Exception as e:
        print(f'\n  ❌ FAIL — decimated capture error: {e}')

## Cell 9 — Noise floor sanity check
With no antenna connected, confirms the ADC noise floor is in a reasonable range.  
**Disconnect any antenna or loopback cable before running this cell.**

In [ ]:
if soc is None:
    print('❌ SKIP — soc not loaded')
else:
    print('  [INFO] Capturing noise floor — antenna should be disconnected\n')

    # Use decimated buffer for speed
    cfg = {'ro_ch':0,'readout_length':1000,'adc_trig_offset':100,
           'soft_avgs':20,'reps':1,'relax_delay':1.0}
    p   = QuickCapture(soc, cfg)
    iq  = p.acquire_decimated(soc, load_pulses=False, progress=False)
    i_d = iq[0][0].astype(np.float32)

    rms     = float(np.sqrt(np.mean(i_d**2)))
    peak    = float(np.max(np.abs(i_d)))
    clip_pc = float(np.mean(np.abs(i_d) > 0.95 * 32767)) * 100

    print(f'  RMS amplitude  : {rms:.2f} ADU')
    print(f'  Peak amplitude : {peak:.2f} ADU  (ADC full scale = 32767)')
    print(f'  Clipping       : {clip_pc:.2f}%')

    if rms < 1000 and peak < 5000 and clip_pc == 0.0:
        print('\n  ✅ PASS — noise floor looks healthy (low level, no clipping)')
    elif clip_pc > 0:
        print('\n  ❌ FAIL — clipping detected with no antenna')
        print('   Check for strong local interference or incorrect gain setting')
    elif rms > 5000:
        print('\n  ⚠️  WARNING — RMS higher than expected for no-antenna noise')
        print('   Some RFI may be coupling in through the board')
    else:
        print('\n  ✅ PASS — noise floor acceptable')

## Cell 10 — Antenna connected: signal sanity check
With antenna connected to ADC_D, confirms real signals are being received.  
**Connect your antenna to ADC_D before running this cell.**

In [ ]:
if soc is None:
    print('❌ SKIP — soc not loaded')
else:
    print('  [INFO] Capturing with antenna — antenna should be on ADC_D\n')

    cfg = {'ro_ch':0,'readout_length':1000,'adc_trig_offset':100,
           'soft_avgs':20,'reps':1,'relax_delay':1.0}
    p   = QuickCapture(soc, cfg)
    iq  = p.acquire_decimated(soc, load_pulses=False, progress=False)
    i_d = iq[0][0].astype(np.float32)

    rms_ant  = float(np.sqrt(np.mean(i_d**2)))
    peak_ant = float(np.max(np.abs(i_d)))
    clip_pc  = float(np.mean(np.abs(i_d) > 0.95 * 32767)) * 100

    # Quick spectrum for sanity
    N   = len(i_d)
    fs  = soc.config['readouts'][0]['fs_decimated']
    win = np.hanning(N)
    spec_db = 10 * np.log10(
        np.abs(np.fft.rfft(i_d * win))**2 / np.sum(win**2) + 1e-30)
    freqs = np.fft.rfftfreq(N, d=1.0/fs)

    peak_freq = float(freqs[np.argmax(spec_db)])
    dynamic_range = float(spec_db.max() - np.median(spec_db))

    print(f'  RMS amplitude  : {rms_ant:.2f} ADU')
    print(f'  Peak amplitude : {peak_ant:.2f} ADU')
    print(f'  Clipping       : {clip_pc:.2f}%')
    print(f'  Spectral peak  : {peak_freq:.1f} MHz at {spec_db.max():.1f} dB')
    print(f'  Dynamic range  : {dynamic_range:.1f} dB above median noise')

    if clip_pc > 5:
        print('\n  ❌ FAIL — heavy clipping detected')
        print('   Add attenuation before connecting LNA')
        print('   Or use soc.set_adc_attenuator() to add digital attenuation')
    elif clip_pc > 0:
        print('\n  ⚠️  WARNING — some clipping detected')
        print('   Monitor carefully when LNA is added')
    elif dynamic_range < 3:
        print('\n  ⚠️  WARNING — very low dynamic range — check antenna connection')
    else:
        print('\n  ✅ PASS — antenna receiving signals, no clipping')

## Cell 11 — ADC attenuator check (RFSoC Gen 3 feature)
The RFSoC 4x2 has a built-in 0–27 dB step attenuator on each ADC.  
This is important when adding the ZKL-2+ LNA (30 dB gain).

In [ ]:
if soc is None:
    print('❌ SKIP — soc not loaded')
else:
    # ADC_D = tile 0, block 0 -> blockname '00'
    BLOCKNAME = '00'

    try:
        current_atten = soc.get_adc_attenuator(BLOCKNAME)
        print(f'  Current ADC attenuator (ADC_D, block {BLOCKNAME}): {current_atten} dB')
        print(f'  Valid range: 0–27 dB in 1 dB steps')
        print(f'  Set with: soc.set_adc_attenuator("{BLOCKNAME}", attenuation)')
        print(f'\n  Recommended settings:')
        print(f'    No LNA         :  0 dB attenuation (current)')
        print(f'    ZKL-2+ LNA     : 20–27 dB attenuation to avoid saturation')
        print(f'    Start with 20 dB and reduce if signal is too weak')
        print('\n  ✅ PASS — ADC attenuator API working')
    except Exception as e:
        print(f'  ❌ FAIL — attenuator read error: {e}')
        print('   This is only available on RFSoC Gen 3 (should work on 4x2)')

## Cell 12 — Disk space check

In [ ]:
import shutil

SAVE_PATH = '/home/xilinx/jupyter_notebooks/spectrum-analyzer/'
os.makedirs(SAVE_PATH, exist_ok=True)

total, used, free = shutil.disk_usage(SAVE_PATH)
free_gb  = free  / 1e9
total_gb = total / 1e9
used_pct = used  / total * 100

# Estimate storage requirement
# One spectrum: 524288 float32 = 2 MB
# At 10 spectra/minute for 24 hours = 14400 spectra = 28.8 GB worst case
# In practice we save every 15 min so much less
SPEC_SIZE_MB    = 524288 * 4 / 1e6
SPECTRA_PER_DAY = 24 * 60 * 4   # ~4 per minute at N_AVG=10
STORAGE_24H_GB  = SPEC_SIZE_MB * SPECTRA_PER_DAY / 1e3

print(f'  Save path      : {SAVE_PATH}')
print(f'  Total disk     : {total_gb:.1f} GB')
print(f'  Used           : {used_pct:.1f}%')
print(f'  Free           : {free_gb:.1f} GB')
print(f'  Estimated need : ~{STORAGE_24H_GB:.1f} GB for 24h (worst case)')

if free_gb > STORAGE_24H_GB * 2:
    print(f'\n  ✅ PASS — plenty of disk space for multi-day observation')
elif free_gb > STORAGE_24H_GB:
    print(f'\n  ⚠️  WARNING — enough for ~1 day but not multi-day')
    print('   Consider reducing FFT size or saving fewer snapshots')
else:
    print(f'\n  ❌ FAIL — insufficient disk space')
    print(f'   Free up space or reduce observation length')

## Cell 13 — RAM availability check

In [ ]:
import subprocess

result = subprocess.run(['cat', '/proc/meminfo'], capture_output=True, text=True)
lines  = {l.split(':')[0].strip(): l.split(':')[1].strip()
          for l in result.stdout.splitlines() if ':' in l}

mem_total_kb = int(lines.get('MemTotal', '0').split()[0])
mem_free_kb  = int(lines.get('MemFree',  '0').split()[0])
mem_avail_kb = int(lines.get('MemAvailable', '0').split()[0])

mem_total_gb = mem_total_kb / 1e6
mem_avail_gb = mem_avail_kb / 1e6

# RAM needed: FFT_SIZE float32 samples + 500 waterfall rows
# = 1M * 4 bytes + 500 * 512K * 4 bytes = 4 MB + 1 GB worst case
FFT_RAM_MB = 1_048_576 * 4 / 1e6
WF_RAM_MB  = 500 * 524288 * 4 / 1e6
TOTAL_NEED_MB = FFT_RAM_MB + WF_RAM_MB + 500  # +500 MB for system

print(f'  Total RAM      : {mem_total_gb:.2f} GB')
print(f'  Available RAM  : {mem_avail_gb:.2f} GB')
print(f'  Estimated need : ~{TOTAL_NEED_MB/1e3:.2f} GB (FFT + waterfall + system)')

if mem_avail_gb * 1e3 > TOTAL_NEED_MB * 1.5:
    print('\n  ✅ PASS — sufficient RAM for full observation')
elif mem_avail_gb * 1e3 > TOTAL_NEED_MB:
    print('\n  ⚠️  WARNING — RAM is tight')
    print('   Reduce MAX_WATERFALL_ROWS to 200 in the survey script')
else:
    print('\n  ❌ FAIL — insufficient RAM')
    print('   Restart the kernel and close all other notebooks before observing')

## Cell 14 — Board temperature check

In [ ]:
try:
    # PYNQ exposes the system monitor for temperature
    from pynq import MMIO
    # Try reading via sysfs — works on PYNQ boards
    temp_files = [
        '/sys/class/thermal/thermal_zone0/temp',
        '/sys/class/thermal/thermal_zone1/temp',
    ]
    temps = []
    for tf in temp_files:
        if os.path.exists(tf):
            with open(tf) as f:
                temps.append(int(f.read().strip()) / 1000)

    if temps:
        for i, t in enumerate(temps):
            print(f'  Thermal zone {i}  : {t:.1f} °C')
        max_temp = max(temps)
        if max_temp < 60:
            print(f'\n  ✅ PASS — board temperature normal ({max_temp:.1f} °C)')
        elif max_temp < 75:
            print(f'\n  ⚠️  WARNING — board warm ({max_temp:.1f} °C)')
            print('   Ensure adequate ventilation for multi-day observations')
        else:
            print(f'\n  ❌ FAIL — board HOT ({max_temp:.1f} °C)')
            print('   Do NOT start a long observation — risk of thermal shutdown')
    else:
        print('  ⚠️  Could not read temperature sensors')
        print('  Ensure the board has ventilation for multi-day runs')

except Exception as e:
    print(f'  ⚠️  Temperature read skipped: {e}')
    print('  Manually check the board is not hot before a long run')

## Cell 15 — Final summary and go/no-go decision

In [ ]:
print('=' * 60)
print('  QICK PRE-FLIGHT CHECKLIST SUMMARY')
print(f'  {datetime.datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S UTC")}')
print('=' * 60)

checks = [
    ('Python imports',         'Run Cell 1  and check for ✅'),
    ('PYNQ version',           'Run Cell 2  — must show 3.0.x'),
    ('QICK version',           'Run Cell 3  — must show 0.2.x'),
    ('Bitstream files',        'Run Cell 4  — both .bit and .hwh'),
    ('Overlay loads',          'Run Cell 5  — soc object created'),
    ('Hardware config',        'Run Cell 6  — ADC/DAC rates correct'),
    ('DDR4 API smoke test',    'Run Cell 7  — arm/get_ddr4 working'),
    ('Decimated buffer',       'Run Cell 8  — acquire_decimated working'),
    ('Noise floor (no ant)',   'Run Cell 9  — low level, no clipping'),
    ('Signal check (ant on)',  'Run Cell 10 — signals visible, no clipping'),
    ('ADC attenuator',         'Run Cell 11 — API working (for LNA use)'),
    ('Disk space',             'Run Cell 12 — enough for observation'),
    ('RAM',                    'Run Cell 13 — enough for waterfall'),
    ('Board temperature',      'Run Cell 14 — below 60°C'),
]

for name, instruction in checks:
    print(f'  [ ] {name:<28} {instruction}')

print()
print('  Spectral resolution summary:')
try:
    fs = soc.config['readouts'][0]['fs']
    print(f'    N=1,048,576  ->  {fs*1e3/1_048_576:.2f} kHz/bin  ✅ below 10 kHz')
    print(f'    N=  524,288  ->  {fs*1e3/524_288:.2f} kHz/bin  ⚠️  borderline')
    print(f'    N=  262,144  ->  {fs*1e3/262_144:.2f} kHz/bin  ❌ above 10 kHz')
except:
    print('    (run Cell 5 first to see computed values)')

print()
print('  Hardware configuration (when ready to observe):')
print('    Antenna       -> ADC_D SMA')
print('    ZKL-2+ LNA    -> between antenna and ADC_D')
print('    Attenuator    -> between LNA and ADC_D if clipping occurs')
print('    Loopback cable -> REMOVE before observation (only for CW tests)')
print()
print('  If all 14 checks pass -> scroll down and run Cell 16')
print('=' * 60)

---
# Observation
## Cell 16 — Configure your observation
**Only run this cell after all 14 preflight checks show ✅.**

| Parameter | Desk antenna (next week) | Discone + LNA (JBO) |
|---|---|---|
| `OBSERVATION_HOURS` | 1–2 | 24–48 |
| `FFT_SIZE` | 1_048_576 | 1_048_576 |
| `N_AVG` | 5 | 20 |
| `ADC_ATTENUATION_DB` | 0 | 20 (with ZKL-2+) |
| `WINDOW_FUNCTION` | hann | hann |

In [ ]:
# ── Edit these values to match your setup ────────────────────────────────

OBSERVATION_HOURS  = 1          # total run time — desk: 1-2h | JBO: 24-48h
FFT_SIZE           = 1_048_576  # 1M -> 4.22 kHz/bin (Phil target <10 kHz)
N_AVG              = 5          # frames per output spectrum — desk: 5 | JBO: 20
WINDOW_FUNCTION    = 'hann'     # 'hann' recommended
ADC_ATTENUATION_DB = 0          # 0 = no LNA | 20 = with ZKL-2+ LNA
SAVE_INTERVAL_MINS = 15         # snapshot every N minutes
MAX_WATERFALL_ROWS = 300        # max rows in RAM
SAVE_PATH          = '/home/xilinx/jupyter_notebooks/spectrum-analyzer/'
ADC_CH             = 0          # readout ch 0 = ADC_D
BLOCKNAME          = '00'       # ADC_D blockname

# ── Computed — do not edit below ─────────────────────────────────────────

FS_MHZ              = soc.config['readouts'][ADC_CH]['fs']
NYQUIST_MHZ         = FS_MHZ / 2
DF_KHZ              = FS_MHZ * 1e3 / FFT_SIZE
SPT                 = DDR4_SAMPLES_PER_TRANSFER if 'DDR4_SAMPLES_PER_TRANSFER' in dir() and DDR4_SAMPLES_PER_TRANSFER else 256
NT                  = (FFT_SIZE // SPT) + 10
OBSERVATION_SECONDS = OBSERVATION_HOURS * 3600

os.makedirs(SAVE_PATH, exist_ok=True)

# Apply ADC attenuation
if ADC_ATTENUATION_DB > 0:
    try:
        soc.set_adc_attenuator(BLOCKNAME, ADC_ATTENUATION_DB)
        actual = soc.get_adc_attenuator(BLOCKNAME)
        print(f'  ✅ ADC attenuator set to {actual} dB')
    except Exception as e:
        print(f'  ⚠️  Could not set attenuator: {e}')
else:
    print('  ℹ️  No ADC attenuation (0 dB)')

# Build window
print(f'  Building {WINDOW_FUNCTION} window for N={FFT_SIZE:,}...')
if WINDOW_FUNCTION == 'hann':
    window = np.hanning(FFT_SIZE).astype(np.float32)
elif WINDOW_FUNCTION == 'blackman':
    window = np.blackman(FFT_SIZE).astype(np.float32)
else:
    window = np.ones(FFT_SIZE, dtype=np.float32)
window_power  = float(np.sum(window**2))
freq_axis_mhz = np.linspace(0, NYQUIST_MHZ, FFT_SIZE // 2,
                             endpoint=False).astype(np.float32)
n_bins = len(freq_axis_mhz)

RFI    = {'FM\n(88-108)': (88,108,'#aaffaa'), 'DAB\n(174-240)': (174,240,'#aaffaa'),
          '4G\n(700-960)': (700,960,'#ffaaff'), 'GPS\n(1575)': (1570,1580,'#aaaaff'),
          '3G/4G\n(2100)': (2100,2170,'#ffaaff')}
WH_FM  = [88.4, 90.4, 94.6, 96.0, 97.6, 99.4, 102.0, 103.0]
WH_DAB = [209.936, 222.064]

print(f'  ✅ Configuration ready')
print(f'\n--- Observation parameters ---')
print(f'  Duration    : {OBSERVATION_HOURS}h')
print(f'  Resolution  : {DF_KHZ:.2f} kHz/bin', '✅ below 10 kHz' if DF_KHZ < 10 else '⚠️  above 10 kHz')
print(f'  Averaging   : {N_AVG} frames/spectrum')
print(f'  Window      : {WINDOW_FUNCTION}')
print(f'  Attenuation : {ADC_ATTENUATION_DB} dB')
print(f'  Nyquist     : {NYQUIST_MHZ:.1f} MHz')
print(f'  DDR4 NT     : {NT} transfers per capture')
print(f'\n  ✅ Ready — run Cell 17 to start')

## Cell 17 — Start observation
**This runs for the full duration set in Cell 16.**  
Progress is printed every output spectrum. Snapshots save every 15 minutes.  
Press ⬛ or Kernel → Interrupt to stop early — data saves automatically.

**Final checklist before running:**
- ✅ Antenna on **ADC_D**
- ✅ Loopback cable **removed**
- ✅ LNA in chain if `ADC_ATTENUATION_DB > 0` was set above
- ✅ Board has ventilation for long runs

In [ ]:
import matplotlib.ticker as ticker
from qick.averager_program import AveragerProgram

class DDR4TriggerProgram(AveragerProgram):
    def initialize(self):
        self.declare_readout(ch=self.cfg['ro_ch'], length=self.cfg['readout_length'],
                             freq=0, gen_ch=None)
        self.synci(200)
    def body(self):
        self.trigger(adcs=[self.cfg['ro_ch']], pins=[0],
                     adc_trig_offset=self.cfg['adc_trig_offset'])
        self.wait_all()
        self.sync_all(self.us2cycles(self.cfg['relax_delay']))

prog = DDR4TriggerProgram(soc, {'ro_ch':ADC_CH,'readout_length':1000,
                                 'adc_trig_offset':100,'soft_avgs':1,
                                 'reps':1,'relax_delay':1.0})

def capture_spectrum():
    soc.arm_ddr4(ch=ADC_CH, nt=NT)
    soc.run_rounds(prog, rounds=1)
    raw    = soc.get_ddr4(ch=ADC_CH, nt=NT, start=None)
    i_data = raw[:, 0].astype(np.float32)
    i_data = i_data[:FFT_SIZE] if len(i_data) >= FFT_SIZE \
             else np.pad(i_data, (0, FFT_SIZE - len(i_data)))
    clip   = float(np.mean(np.abs(i_data) > 0.95 * 32767))
    spec   = np.abs(np.fft.rfft(i_data * window, n=FFT_SIZE))**2 / window_power
    return 10 * np.log10(spec[:FFT_SIZE//2] + 1e-30).astype(np.float32), clip

def save_plot(ts, wf, mh, ms, ns, elapsed, clip_pct):
    fig, axes = plt.subplots(2, 1, figsize=(18, 10))
    fig.patch.set_facecolor('#0d0d0d')
    ax1, ax2  = axes
    ax1.set_facecolor('#0d0d0d')
    ax1.plot(freq_axis_mhz, ms,  color='#00e5ff', lw=0.4, alpha=0.8, label='Mean')
    ax1.plot(freq_axis_mhz, mh,  color='#ff6b35', lw=0.5, label='Max hold')
    for lbl,(f0,f1,col) in RFI.items():
        if f0 < float(freq_axis_mhz[-1]):
            ax1.axvspan(f0,min(f1,float(freq_axis_mhz[-1])),alpha=0.08,color=col)
            ax1.text((f0+min(f1,float(freq_axis_mhz[-1])))/2, mh.max()+1,
                     lbl, color=col, fontsize=5, ha='center', va='bottom')
    for f in WH_FM:  ax1.axvline(f,color='#88ff88',lw=0.5,ls='--',alpha=0.5)
    ax1.axvline(WH_FM[0],color='#88ff88',lw=0.5,ls='--',alpha=0.5,label='Winterhill FM')
    for f in WH_DAB: ax1.axvline(f,color='#ffff88',lw=0.5,ls=':',alpha=0.5)
    ax1.axvline(WH_DAB[0],color='#ffff88',lw=0.5,ls=':',alpha=0.5,label='Winterhill DAB')
    ax1.set_xlim(0, NYQUIST_MHZ)
    ax1.set_ylabel('Power (dB)', color='white', fontsize=11)
    ax1.set_title(f'QICK DDR4 RFI Monitor | RFSoC 4x2 | UTC {ts} | '
                  f'{elapsed/3600:.2f}h | {ns} spectra | '
                  f'{DF_KHZ:.2f} kHz/bin | Clip: {clip_pct:.2f}%',
                  color='white', fontsize=9)
    ax1.tick_params(colors='white'); ax1.spines[:].set_color('#333333')
    ax1.grid(True, color='#1e1e1e', lw=0.3)
    ax1.xaxis.set_major_locator(ticker.MultipleLocator(200))
    ax1.legend(facecolor='#1a1a1a',edgecolor='#444',labelcolor='white',
               fontsize=7,loc='upper right')
    ax2.set_facecolor('#0d0d0d')
    if len(wf) > 1:
        im = ax2.imshow(wf, aspect='auto',
                        extent=[0,NYQUIST_MHZ,elapsed/3600,0], cmap='inferno',
                        vmin=np.percentile(wf,2), vmax=np.percentile(wf,98))
        cb = plt.colorbar(im, ax=ax2, label='Power (dB)', pad=0.01)
        cb.ax.yaxis.label.set_color('white'); cb.ax.tick_params(colors='white')
        for _,(f0,f1,col) in RFI.items():
            if f0 < float(freq_axis_mhz[-1]):
                ax2.axvspan(f0,min(f1,float(freq_axis_mhz[-1])),alpha=0.06,color=col)
    ax2.set_xlabel('Frequency (MHz)',color='white',fontsize=11)
    ax2.set_ylabel('Time (hours)',color='white',fontsize=11)
    ax2.set_title(f'Waterfall ({len(wf)} rows | {DF_KHZ:.2f} kHz/bin | '
                  f'{WINDOW_FUNCTION} window)', color='white', fontsize=9)
    ax2.tick_params(colors='white')
    ax2.xaxis.set_major_locator(ticker.MultipleLocator(200))
    plt.tight_layout()
    out = f'{SAVE_PATH}qick_ddr4_survey_{ts}.png'
    fig.savefig(out, dpi=150, bbox_inches='tight', facecolor='#0d0d0d')
    plt.close(fig)
    return out

# ── Acquisition loop ──────────────────────────────────────────────────────
ts = datetime.datetime.utcnow().strftime('%Y%m%d_%H%M%S')
max_hold = np.full(n_bins, -200.0, dtype=np.float32)
waterfall, frame_acc = [], np.zeros(n_bins, dtype=np.float64)
n_frames_acc = n_spectra = n_clips = n_total = consecutive_errors = 0
t_start = t_last_save = time.time()

print(f'[OBS] Started UTC {ts}')
print(f'[OBS] {OBSERVATION_HOURS}h | {DF_KHZ:.2f} kHz/bin | '
      f'{N_AVG} frames/spectrum | {WINDOW_FUNCTION}')
print(f'[OBS] Press ⬛ Stop or Kernel → Interrupt to stop early\n')

try:
    while (time.time() - t_start) < OBSERVATION_SECONDS:
        try:
            spec_db, clip_frac = capture_spectrum()
            consecutive_errors = 0
        except Exception as e:
            consecutive_errors += 1
            print(f'  [WARN] Error {consecutive_errors}/5: {e}')
            if consecutive_errors >= 5:
                print('  [STOP] Too many errors — aborting'); break
            time.sleep(5); continue

        n_total += 1
        if clip_frac > 0.01:
            n_clips += 1
            if n_clips % 10 == 1:
                print(f'  [CLIP] ⚠️  {clip_frac*100:.1f}% clipping — '
                      f'add attenuation')

        frame_acc += spec_db.astype(np.float64)
        n_frames_acc += 1

        if n_frames_acc >= N_AVG:
            averaged     = (frame_acc / n_frames_acc).astype(np.float32)
            max_hold     = np.maximum(max_hold, averaged)
            if len(waterfall) >= MAX_WATERFALL_ROWS: waterfall.pop(0)
            waterfall.append(averaged.copy())
            n_spectra   += 1; frame_acc[:] = 0.0; n_frames_acc = 0
            elapsed   = time.time() - t_start
            clip_pct  = n_clips / n_total * 100
            print(f'  #{n_spectra:5d} | frames {n_total:6d} | '
                  f'{elapsed/3600:5.2f}h / {OBSERVATION_HOURS}h | '
                  f'clip {clip_pct:.2f}% | '
                  f'peak {averaged.max():.1f} dB | '
                  f'noise {np.median(averaged):.1f} dB')

        if (time.time() - t_last_save) >= SAVE_INTERVAL_MINS * 60:
            elapsed  = time.time() - t_start
            clip_pct = n_clips / n_total * 100 if n_total else 0
            wf_array = np.array(waterfall, dtype=np.float32)
            ms       = np.mean(wf_array, axis=0) if len(wf_array) > 0 else max_hold
            np.save(f'{SAVE_PATH}qick_ddr4_maxhold_{ts}.npy',   max_hold)
            np.save(f'{SAVE_PATH}qick_ddr4_waterfall_{ts}.npy', wf_array)
            np.save(f'{SAVE_PATH}qick_ddr4_freqaxis_{ts}.npy',  freq_axis_mhz)
            pp = save_plot(ts, wf_array, max_hold, ms,
                           n_spectra, elapsed, clip_pct)
            snap = datetime.datetime.utcnow().strftime('%H:%M:%S')
            print(f'\n  [SAVE] {snap} — {n_spectra} spectra | '
                  f'{os.path.basename(pp)}\n')
            t_last_save = time.time()

except KeyboardInterrupt:
    print('\n[STOP] Interrupted — saving final result...')

# ── Final save ────────────────────────────────────────────────────────────
elapsed_total = time.time() - t_start
clip_fraction = n_clips / n_total * 100 if n_total > 0 else 0
wf_array  = np.array(waterfall, dtype=np.float32)
mean_spec = np.mean(wf_array, axis=0) if len(wf_array) > 0 else max_hold
np.save(f'{SAVE_PATH}qick_ddr4_maxhold_{ts}.npy',   max_hold)
np.save(f'{SAVE_PATH}qick_ddr4_waterfall_{ts}.npy', wf_array)
np.save(f'{SAVE_PATH}qick_ddr4_freqaxis_{ts}.npy',  freq_axis_mhz)
pp = save_plot(ts, wf_array, max_hold, mean_spec,
               n_spectra, elapsed_total, clip_fraction)

print(f'\n[DONE]')
print(f'  Duration    : {elapsed_total/3600:.2f}h')
print(f'  Frames      : {n_total}')
print(f'  Spectra     : {n_spectra}')
print(f'  Clipping    : {clip_fraction:.2f}%')
print(f'  Resolution  : {DF_KHZ:.2f} kHz/bin')
print(f'\nDownload with:')
print(f'  scp xilinx@192.168.3.1:{pp} .')
print(f'  scp xilinx@192.168.3.1:{SAVE_PATH}qick_ddr4_*_{ts}.npy .')